# PA1 — Segmentação de instâncias · execução completa

Sobe este notebook no Colab e roda **Executar tudo**. Ele clona o repositório,
treina tudo do zero com o split estratificado e gera todos os resultados e
figuras das Partes 0 a 7.

**Antes de começar:** Ambiente de execução → Alterar o tipo de ambiente de
execução → **GPU (T4)**.

**Tempo:** cerca de 3 horas, quase tudo nos 16 treinos das Partes 1 e 3.
O Colab derruba sessões ociosas — deixe a aba aberta.

Nenhuma célula pede interação, então dá para rodar tudo e voltar depois.

## 1. Ambiente

In [ ]:
import os

if not os.path.exists('/content/deep-learning-2026.2'):
    !git clone https://github.com/sofiaazeredo/deep-learning-2026.2.git

%cd /content/deep-learning-2026.2
!git checkout structure && git pull
%cd /content/deep-learning-2026.2/PA1

!pwd && git branch --show-current

In [ ]:
import torch, pathlib

print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available()
       else "NENHUMA -> troque o ambiente de execucao para GPU antes de continuar")
print("amostras:", len(list(pathlib.Path("data/raw").iterdir())))

from src.model import UNet, UNetNoSkips, UNetASPP
from src.dataset import DSB2018Dataset, create_splits, split_report

ds = DSB2018Dataset("data/raw")
print()
print("Split estratificado por modalidade (a justificativa que o enunciado pede):")
split_report(ds, create_splits(ds, seed=42, train_ratio=0.8, val_ratio=0.1))

## 2. Backup no Drive — para a queda do Colab não custar nada

Autorize o acesso quando o Colab pedir. É o único clique do notebook inteiro.

Cada treino vai para o Drive assim que termina. Se a sessão morrer, **rode este
mesmo notebook de novo**: ele restaura o que já foi feito e pula direto para o
que falta.


In [ ]:
import shutil
from pathlib import Path

BACKUP = None

try:
    from google.colab import drive
    drive.mount('/content/drive')
    BACKUP = Path('/content/drive/MyDrive/PA1_backup')
    (BACKUP / 'checkpoints').mkdir(parents=True, exist_ok=True)
    (BACKUP / 'results').mkdir(parents=True, exist_ok=True)
    print('backup em:', BACKUP)
except Exception as erro:
    print('SEM BACKUP --', erro)
    print('Se a maquina cair, perde tudo. Rode esta celula de novo para tentar.')


# Traz de volta o que ja foi treinado numa sessao anterior.
def restaurar():
    if BACKUP is None:
        return
    total = 0
    pares = ((BACKUP / 'checkpoints', Path('checkpoints')),
             (BACKUP / 'results', Path('experiments/results')))
    for origem, destino in pares:
        destino.mkdir(parents=True, exist_ok=True)
        for arquivo in origem.glob('*'):
            alvo = destino / arquivo.name
            if arquivo.is_file() and not alvo.exists():
                shutil.copy2(arquivo, alvo)
                total += 1
    print('restaurados', total, 'arquivos do Drive')


# Copia checkpoint e CSVs para o Drive.
def salvar(nome=None):
    if BACKUP is None:
        return
    if nome:
        peso = Path('checkpoints') / (nome + '_best.pt')
        if peso.exists():
            shutil.copy2(peso, BACKUP / 'checkpoints' / peso.name)
    for arquivo in Path('experiments/results').glob('*'):
        if arquivo.is_file():
            shutil.copy2(arquivo, BACKUP / 'results' / arquivo.name)


# Treino + avaliacao ja concluidos para este experimento?
def ja_feito(nome):
    return Path('experiments/results/' + nome + '_test_metrics.csv').exists()


restaurar()


## 3. Parte 0 — teste unitário sintético

Não depende dos dados reais nem do split. Roda primeiro justamente por isso: se
alguma peça do pipeline estiver quebrada, quebra aqui em 3 minutos em vez de
depois de 3 horas de treino.

In [ ]:
!python scripts/train_synthetic.py
salvar()


## 4. Parte 1 — baseline semântico + instâncias ingênuas

In [ ]:
from pathlib import Path

if Path('experiments/results/baseline_test_metrics.csv').exists():
    print('ja feito, pulando')
else:
    !python scripts/train.py
    !python scripts/evaluate.py

!python scripts/plot_baseline_results.py
salvar('baseline')


## 5. Partes 2 e 3 (Eixo 2) — fronteira + watershed, ablação de perdas

10 treinos: 5 perdas × 2 seeds. O braço `balanced_ce` é também o modelo da
Parte 2 e o braço "U-Net + skips" do Eixo 1.

In [ ]:
CONFIGS = [
    ('--loss ce',               'loss_ce'),
    ('--loss balanced_ce',      'loss_balanced'),
    ('--loss focal --gamma 1',  'loss_focal1'),
    ('--loss focal --gamma 2',  'loss_focal2'),
    ('--loss focal --gamma 5',  'loss_focal5'),
]

for flags, prefixo in CONFIGS:
    for seed in (42, 123):
        nome = prefixo + '_seed' + str(seed)
        if ja_feito(nome):
            print('###', nome, 'ja feito, pulando')
            continue
        print('\n' + '='*70 + '\n### ' + nome + '\n' + '='*70)
        !python scripts/train_boundary.py {flags} --seed {seed} --epochs 20 --name {nome}
        !python scripts/evaluate_boundary.py --checkpoint checkpoints/{nome}_best.pt --name {nome}
        salvar(nome)


## 6. Parte 3 (Eixo 1) — recuperação de resolução

4 treinos: 2 arquiteturas × 2 seeds, mesma perda e mesmo split.

In [ ]:
for arquitetura in ('no_skips', 'aspp'):
    for seed in (42, 123):
        nome = 'resolution_' + arquitetura + '_seed' + str(seed)
        if ja_feito(nome):
            print('###', nome, 'ja feito, pulando')
            continue
        print('\n' + '='*70 + '\n### ' + nome + '\n' + '='*70)
        !python scripts/train_boundary.py --architecture {arquitetura} --loss balanced_ce --seed {seed} --epochs 20 --name {nome}
        !python scripts/evaluate_boundary.py --checkpoint checkpoints/{nome}_best.pt --name {nome}
        salvar(nome)


## 7. Ablações agregadas e escolha do modelo final

In [ ]:
!python scripts/summarize_loss_ablation.py
!python scripts/summarize_resolution_ablation.py

import json
melhor = json.load(open('experiments/results/best_architecture.json'))
print()
print('MODELO FINAL:', melhor['config'], '->', melhor['checkpoint'])
salvar()


## 8. Partes 4, 5 e 6

Só inferência, alguns minutos. Os scripts leem `best_architecture.json` e
resolvem o checkpoint vencedor sozinhos.

In [ ]:
!python scripts/mosaic_inference.py --grid 3 --name mosaic
!python scripts/receptive_field.py
!python scripts/failure_gallery.py --n-failures 5 --name failures
!python scripts/failure_correction.py --name correcao
salvar()


In [ ]:
!python scripts/stress_test.py --mode corruptions
!python scripts/stress_test.py --mode scale
salvar()


## 9. Parte 7 — inferência numa imagem qualquer, sem retreinar

In [ ]:
from pathlib import Path
import matplotlib.pyplot as plt
from src.inference import segment_image

image_path = next((sorted(Path("data/raw").iterdir())[0] / "images").iterdir())

result = segment_image(image_path)

print("checkpoint:", result["checkpoint"])
print("OBJETOS DETECTADOS:", result["count"])

fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes[0].imshow(result["image"]);   axes[0].set_title("imagem")
axes[1].imshow(result["colored"]); axes[1].set_title(f"instâncias ({result['count']})")
axes[2].imshow(result["overlay"]); axes[2].set_title("sobreposição")
for ax in axes: ax.axis("off")
plt.show()

## 10. Resumo de tudo que foi gerado

In [ ]:
import pandas as pd
from pathlib import Path

print("=" * 70)
print("RESULTADOS FINAIS")
print("=" * 70)

sint = pd.read_csv("experiments/results/synthetic_summary.csv").set_index("metric")["value"]
print(f"\nParte 0 (sintético): mAP {float(sint['map_50_95']):.4f} "
      f"em {float(sint['training_time_s']):.0f}s")

base = pd.read_csv("experiments/results/baseline_summary.csv").iloc[0]
print(f"Parte 1 (baseline):  IoU {base['semantic_iou']:.4f} · "
      f"Dice {base['semantic_dice']:.4f} · mAP {base['map_50_95']:.4f}")

print("\nParte 3 — Eixo 2 (perdas):")
print(pd.read_csv("experiments/results/loss_ablation_summary.csv")
        [["config","map_mean","map_std"]].to_string(index=False))

print("\nParte 3 — Eixo 1 (resolução):")
print(pd.read_csv("experiments/results/resolution_ablation_summary.csv")
        [["config","map_mean","map_std","count_error_mean"]].to_string(index=False))

print("\nParte 4 — mosaico:")
print(pd.read_csv("experiments/results/mosaic_results.csv").to_string(index=False))

print("\nParte 6 — estresse (corrupções):")
print(pd.read_csv([str(p) for p in Path("experiments/results").glob("stress_corruptions_*.csv")][0])
        .to_string(index=False))

print(f"\n{len(list(Path('experiments/results').glob('*.csv')))} CSVs · "
      f"{len(list(Path('experiments/figures').rglob('*.png')))} figuras")

## 11. Salvar os resultados

Os resultados vivem só nesta máquina virtual: se ela reciclar, some tudo.
Preencha o token e rode para mandar CSVs e figuras para o GitHub.

Criar um token: GitHub → Settings → Developer settings → Personal access tokens
→ Fine-grained tokens, com permissão de escrita neste repositório.

In [ ]:
TOKEN = ""        # cole aqui
EMAIL = "bryan.monteiro@dharma-ai.com"
NOME  = "bryanmonteiro"

if TOKEN:
    !git add experiments/results/ experiments/figures/
    !git -c user.email="{EMAIL}" -c user.name="{NOME}" commit -m "resultados com split estratificado"
    !git push https://{TOKEN}@github.com/sofiaazeredo/deep-learning-2026.2.git structure
else:
    print("Sem token: baixando um zip com os resultados.")
    !zip -qr /content/resultados_pa1.zip experiments/
    from google.colab import files
    files.download("/content/resultados_pa1.zip")

## 12. Checkpoint final

O modelo final é o único peso que vai para a entrega. Baixe e suba no Drive,
depois atualize o link no `README.md`.

In [ ]:
import json
best = json.load(open("experiments/results/best_architecture.json"))["checkpoint"]
print("checkpoint final:", best)
!ls -lh {best}

from google.colab import files
files.download(best)